In [3]:
# ==========================================
# STEP 1: INSTALL LIBRARIES
# ==========================================
# easyocr: Reads the text
# wordsegment: Splits merged words (e.g. "SurfaceChemistry") automatically
# pyspellchecker: Fixes typos based on math, not rules
!pip install easyocr python-docx wordsegment pyspellchecker

import easyocr
import cv2
import re
import numpy as np
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from google.colab import files
from wordsegment import load, segment
from spellchecker import SpellChecker

# Initialize NLP Libraries
load() # Load the massive English corpus for word segmentation
spell = SpellChecker()

# ==========================================
# STEP 2: GENERIC PREPROCESSING
# ==========================================
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Standard adaptive thresholding to clean paper texture
    # This is generic computer vision, not hardcoding
    clean = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 15
    )
    return clean, img.shape[1]

# ==========================================
# STEP 3: ALGORITHMIC TEXT REPAIR
# ==========================================
def repair_text_with_nlp(text):
    # 1. Remove non-standard symbols (noise) using Regex
    # Keeps letters, numbers, spaces, and basic punctuation
    clean_chars = re.sub(r'[^a-zA-Z0-9\s\.,\-\(\)]', ' ', text)

    # 2. Split text into words
    words = clean_chars.split()
    final_words = []

    for word in words:
        # 3. Check for Merged Words (e.g., "SurfaceChemistry")
        # If a word is long (>10 chars) or mixed case, try to segment it
        if len(word) > 10 or not word.isalpha():
            # wordsegment library automatically finds the best split
            # e.g. "surfacechemistry" -> ["surface", "chemistry"]
            segmented = segment(word)
            final_words.extend(segmented)
        else:
            # 4. Standard Spell Check
            # If the word is unknown, find the mathematically closest correction
            if word.lower() not in spell:
                correction = spell.correction(word)
                # If correction found, use it; otherwise keep original
                final_words.append(correction if correction else word)
            else:
                final_words.append(word)

    return " ".join(final_words)

# ==========================================
# STEP 4: MAIN EXECUTION
# ==========================================
reader = easyocr.Reader(['en'])

print("\n=== UNIVERSAL CONVERTER (NLP BASED) ===")
print("Upload your image...")
uploaded = files.upload()

if uploaded:
    image_path = list(uploaded.keys())[0]

    # 1. Clean Image
    clean_img, width = preprocess_image(image_path)
    cv2.imwrite("temp_ocr_input.jpg", clean_img)

    # 2. Read Text
    print("Reading text...")
    results = reader.readtext("temp_ocr_input.jpg")

    # 3. Sort & Group Lines (Standard Layout Analysis)
    results.sort(key=lambda x: x[0][0][1])
    lines_grouped = []
    current_group = []

    if results:
        current_group.append(results[0])
        for i in range(1, len(results)):
            box, text, conf = results[i]
            prev_box = current_group[-1][0]
            # Group text that is vertically close (< 20 pixels)
            if abs(box[0][1] - prev_box[0][1]) < 20:
                current_group.append(results[i])
            else:
                lines_grouped.append(current_group)
                current_group = [results[i]]
        lines_grouped.append(current_group)

    # 4. Generate Document
    print(f"Processing {len(lines_grouped)} lines...")
    doc = Document()
    doc.add_heading('Converted Output', 0)

    for group in lines_grouped:
        # Sort words Left -> Right
        group.sort(key=lambda x: x[0][0][0])

        # Combine Raw Text
        raw_text = " ".join([item[1] for item in group])

        # Apply NLP Repair
        final_text = repair_text_with_nlp(raw_text)

        # Output to user
        print(f"Fixed: {final_text}")

        p = doc.add_paragraph()
        run = p.add_run(final_text)

        # Geometric Alignment (Math-based, not rule-based)
        first_box = group[0][0] # Top-Left of first word
        last_box = group[-1][0] # Bottom-Right of last word

        # Calculate center of the text line
        line_center = (first_box[0][0] + last_box[1][0]) / 2

        if (width * 0.35) < line_center < (width * 0.65):
            p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            if len(final_text) < 50: run.bold = True
        elif first_box[0][0] > (width * 0.6):
            p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        else:
            p.alignment = WD_ALIGN_PARAGRAPH.LEFT

    filename = "Universal_NLP_Output.docx"
    doc.save(filename)
    print(f"\nSuccess! Download: {filename}")
    files.download(filename)

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete
=== UNIVERSAL CONVERTER (NLP BASED) ===
Upload your image...


Saving img1.jpeg to img1.jpeg
Reading text...
Processing 26 lines...
Fixed: 1 SURFAcE CUcMsTRY
Fixed: devotion jc cumu lution j molecular spec les of the serfage ba in Ian
Fixed: 77gaqsonbofe H ga dean ten bulk solid e 08 uni d soda Descepkop odsoabote the process job of remove of asben
Fixed: soul stop
Fixed: enthalpy can we c exo the amic Then odom molecule ton the
Fixed: YCnhopu 4s we ah t4s the bulk 0f sold 0 liquid d
Fixed: gibbs five ene8dcageve ed ho wp absorbent be cache
Fixed: type oaclsougtioo solon
Fixed: physical oct so xp lion handle escxllls voice valet chemtauods all oo jso ip lion q obs ovp lion simuyfoneouyla 2 dying f fabric
Fixed: reversible joyce iz reve a sible
Fixed: hysIcclac sorption Y a 4 plus activation lis n pexjicaa late o gganzva Tc kuquijiculion U specific 4tc geliirpraeptan join b0 e thai tc a
Fixed: r7rb
Fixed: foctoxdhecioq as oop to q rest 0 temp on acjsdab lion
Fixed: cause of 186k pkozodtx eeoapbox5 idjouphonqadtc has io soap tion deep
Fixed: na tute

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==========================================
# 1. INSTALL DEPENDENCIES
# ==========================================
!pip install easyocr python-docx wordsegment pyspellchecker gradio

import easyocr
import cv2
import re
import os
import gradio as gr
from docx import Document
from wordsegment import load, segment
from spellchecker import SpellChecker

# Initialize NLP
load()
spell = SpellChecker()
reader = easyocr.Reader(['en'])

# ==========================================
# 2. THE PROCESSING LOGIC (NLP-Based)
# ==========================================
def process_handwriting(input_img):
    if input_img is None:
        return None, "Please upload an image first."

    # Convert Gradio image (numpy) to Gray for OCR
    gray = cv2.cvtColor(input_img, cv2.COLOR_BGR2GRAY)
    clean = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)
    cv2.imwrite("temp.jpg", clean)

    # Run OCR
    results = reader.readtext("temp.jpg")
    results.sort(key=lambda x: x[0][0][1]) # Sort vertically

    doc = Document()
    doc.add_heading('Handwritten Notes Conversion', 0)

    for (_, text, _) in results:
        # Algorithmic Repair (No Hallucinations)
        clean_text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
        words = clean_text.split()
        repaired = []

        for word in words:
            if len(word) > 10:
                repaired.extend(segment(word))
            else:
                corr = spell.correction(word)
                repaired.append(corr if corr else word)

        final_line = " ".join(repaired)
        if len(final_line) > 2:
            doc.add_paragraph(final_line)

    output_path = "Converted_Notes.docx"
    doc.save(output_path)

    return output_path, "Conversion Successful! Download your file below."

# ==========================================
# 3. THE GUI (Gradio) & DEPLOYMENT
# ==========================================
interface = gr.Interface(
    fn=process_handwriting,
    inputs=gr.Image(label="Upload Handwritten Note"),
    outputs=[
        gr.File(label="Download Word Document"),
        gr.Textbox(label="Status")
    ],
    title="Handwriting to Word Converter",
    description="Upload a photo of your notes. This app uses NLP to split merged words and fix typos without inventing fake text.",
    theme="soft"
)

# share=True creates the public deployment link
interface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://68537065a5b292ce29.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
